## Goals

- Now, we can extend this into generic agent architecture.
- In the router, we invoked the model and if it chose to call a tool, we returned a ToolMessage to the user
- But, What if we simply pass that ToolMessage back to the model?
- We can let it either (1) call another tool or (2) respond directly.

This is the intuition [ReAct](https://react-lm.github.io/), a general agent architecture:
- act: Let the model call specific tools
- observe: pass the tool output back to the model
- reason: let the model reason about the tool output to decide what to do next (Call another tool or just respond directly)

This [general purpose architecture](https://www.langchain.com/blog/planning-for-agents) can be applied to many types of tools





In [4]:
import os
import json
from dotenv import load_dotenv

from langchain_ollama import ChatOllama


In [3]:
load_dotenv(override=True)

True

In [ ]:
qwen = ChatOllama(model="qwen2.5:7b", temperature=0)

## Define tools

In [6]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

# This will be a tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

In [8]:
tools = [multiply, add, divide]

llm_with_tools = qwen.bind_tools(tools, parallel_tool_calls=False)

In [9]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node

def assistant(state: MessagesState):
    return {
        "messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]
    }

- As before, we use MessagesState and define a Tools node with our list of tools.

- The Assistant node is just our model with bound tools.

- We create a graph with **Assistant** and **Tools** nodes.

- We add **tools_condition** edge, which routes to End or to Tools based on whether the Assistant calls a tool.

Now, we add one new step:

- We connect the Tools node back to the Assistant, forming a loop.
    - After the assistant node executes, tools_condition checks if the model's output is a tool call.
    - If it is a tool call, the flow is directed to the tools node.
    - The tools node connects back to assistant.
    - This loop continues as long as the model decides to call tools.
    - If the model response is not a tool call, the flow is directed to END, terminating the process.
